In [ ]:
"""
Experiment 6: Intertemporal feature attribution analysis.
The dataset is divided into:
    - early period (first 11 years)
    - later period (last 11 years)
Results are aggregated across five cross-validation folds.
Purpose: Evaluate the robustness of feature importance across time periods.
"""

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split, KFold
from tqdm import tqdm
import function
os.makedirs('../results/work/6', exist_ok=True)

### Load datasets.
yield_data = pd.read_csv('../data/yield.csv', engine='python').values[:,1:]
social_data = pd.read_csv('../data/social.csv', engine='python').values[:,1:]
natural_dataset = pd.read_csv('../data/natural.csv', engine='python')
natural_data = function.natural_feature_builder(natural_dataset).values

### Convert data to tensor format and reshape into county × year × feature structure.
yield_data = torch.from_numpy(yield_data.astype('float')).float()
social_data = torch.from_numpy(social_data.astype('float')).float().reshape(-1,22,9)
natural_data = torch.from_numpy(natural_data.astype('float')).float().reshape(22,12,-1,8)[:,:9].permute(2,0,1,3).reshape(654,22,-1)

### Filter, resolve, and flatten datasets using custom functions.
county_index = np.arange(654).reshape(-1,1)
yield_data, social_data, natural_data, county_index = function.filter_(yield_data, social_data, natural_data, county_index)
yield_tear, social_tear, natural_tear, county_index = function.resolve(yield_data, social_data, natural_data, county_index)
yield_obsvd, social_obsvd, natural_obsvd = yield_tear[:,0].reshape(-1,1), social_tear[0], natural_tear[0]
yield_resid, social_resid, natural_resid = yield_tear[:,0].reshape(-1,1), social_tear[2], natural_tear[2]

### Train/test split and K-fold cross validation for each time period.
n_splits = 5
for i in range(2):
    p, index = ['early', 'later'][i], [np.arange(11), np.arange(11, 22)][i]
    index_index = np.isin(county_index[:,1], index)
    exec(f"yield_obsvd_{p}, yield_resid_{p} = yield_obsvd[index_index], yield_resid[index_index]")
    exec(f"social_obsvd_{p}, social_resid_{p} = social_obsvd[index_index], social_resid[index_index]")
    exec(f"natural_obsvd_{p}, natural_resid_{p} = natural_obsvd[index_index], natural_resid[index_index]")
    exec(f"data_index = np.arange(yield_obsvd_{p}.size(0))")
    train_valid_index, test_index = train_test_split(data_index, test_size=0.15)
    kf, kf_indices =  KFold(n_splits=n_splits, shuffle=True), []
    for train_index, valid_index in kf.split(train_valid_index):
        kf_indices.append((train_valid_index[train_index], train_valid_index[valid_index]))
    exec(f"kf_indices_{p}, test_index_{p}, county_index_{p} = kf_indices, test_index, county_index[index_index]")

In [ ]:
### Train models and record feature attribution.
model_iter, num_instance_identity, require_explanation, batch_size, learning_rate, max_epoch, patience, train_breaker =\
    ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att'], 128, True, 32, 0.0003, 1000, 20, False
social_columns = ['population_size', 'health-care_resource', 'pupil_proportion', 'teenager_proportion', 
                  'fiscal_revenue', 'fiscal_expenditure', 'residential_savings', 'institutional_loans']
natural_columns = ['radiation', 'air_temperature', 'soil_temperature', 'air_humidity',
                   'soil_moisture', 'precipitation', 'surface_pressure', 'wind_speed']
feat_columns = ['county', 'yield'] + social_columns
for i in range(9):
    for j in natural_columns:
        exec(f"feat_columns.append('{j}{i+1}')")
for p in tqdm(['early', 'later']):
    for i in ['obsvd', 'resid']:
        exec(f"county_index, test_index, kf_indices = county_index_{p}, test_index_{p}, kf_indices_{p}")
        exec(f"yield_data, social_data, natural_data = yield_{i}_{p}, social_{i}_{p}, natural_{i}_{p}")
        for model in tqdm(model_iter):
            for j in range(n_splits):
                train_index, valid_index = kf_indices[j][0], kf_indices[j][1]
                yield_train, social_train, natural_train = yield_data[train_index], social_data[train_index], natural_data[train_index]
                yield_valid, social_valid, natural_valid = yield_data[valid_index], social_data[valid_index], natural_data[valid_index]
                yield_test, social_test, natural_test = yield_data[test_index], social_data[test_index], natural_data[test_index]
                yield_train_scaled, yield_valid_scaled, yield_test_scaled = function.scaler(yield_train, yield_valid, yield_test)
                social_train_scaled, social_valid_scaled, social_test_scaled = function.scaler(social_train, social_valid, social_test)
                natural_train_scaled, natural_valid_scaled, natural_test_scaled = function.scaler(natural_train, natural_valid, natural_test)
                output_kf = function.predictor(model, 'addition', num_instance_identity, require_explanation,
                                            yield_train_scaled, social_train_scaled, natural_train_scaled,
                                            yield_valid_scaled, social_valid_scaled, natural_valid_scaled,
                                            batch_size, learning_rate, max_epoch, patience, train_breaker,
                                            yield_test_scaled, social_test_scaled, natural_test_scaled)
                feat_ig, feat_gs = output_kf[4][0], output_kf[4][1]
                feat_ig = np.concatenate([county_index[test_index,0].reshape(-1,1), yield_test, feat_ig], 1)
                feat_gs = np.concatenate([county_index[test_index,0].reshape(-1,1), yield_test, feat_gs], 1)
                exec(f"feat_ig_{model}_kf{j+1}_csv = pd.DataFrame(feat_ig, columns=feat_columns)")
                exec(f"feat_gs_{model}_kf{j+1}_csv = pd.DataFrame(feat_gs, columns=feat_columns)")

        for j in model_iter:
            exec(f"feat_ig_{j}, feat_gs_{j} = [], []")
        for j in range(n_splits):
            for k in model_iter:
                exec(f"feat_ig_{k}.append(feat_ig_{k}_kf{j+1}_csv.values)")
                exec(f"feat_gs_{k}.append(feat_gs_{k}_kf{j+1}_csv.values)")
        for j in model_iter:
            exec(f"feat_ig, feat_gs = np.concatenate(feat_ig_{j}), np.concatenate(feat_gs_{j})")
            exec(f"feat_ig_{j}_csv = pd.DataFrame(feat_ig[np.lexsort((feat_ig[:,1],feat_ig[:,0]))], columns=feat_columns)")
            exec(f"feat_gs_{j}_csv = pd.DataFrame(feat_gs[np.lexsort((feat_gs[:,1],feat_gs[:,0]))], columns=feat_columns)")
        index = [pd.DataFrame(test_index.reshape(-1,1))]
        for j in range(n_splits):
            index.append(pd.DataFrame(kf_indices[j][0]))
            index.append(pd.DataFrame(kf_indices[j][1]))
        index_csv = pd.concat(index, 1)
        index_csv.columns = ['test', 'kf1_t', 'kf1_v', 'kf2_t', 'kf2_v', 'kf3_t', 'kf3_v', 'kf4_t', 'kf4_v', 'kf5_t', 'kf5_v']

### Save aggregated results.
        exec(f"index_csv.to_csv('../results/work/6/{p}_index.csv', index=False)")
        for j in model_iter:
            exec(f"feat_ig_{j}_csv.to_csv('../results/work/6/{p}_{i}_ig_{j}.csv', index=False)")
            exec(f"feat_gs_{j}_csv.to_csv('../results/work/6/{p}_{i}_gs_{j}.csv', index=False)")
            for k in range(n_splits):
                exec(f"feat_ig_{j}_kf{k+1}_csv.to_csv('../results/work/6/{p}_{i}_ig_{j}_kf{k+1}.csv', index=False)")
                exec(f"feat_gs_{j}_kf{k+1}_csv.to_csv('../results/work/6/{p}_{i}_gs_{j}_kf{k+1}.csv', index=False)")